## PropertyLens RAG — Notebook B: Inference (v4.1)

**Purpose:** query-time pipeline. Connects to a populated Pinecone index, runs the full V3 retrieval stack (hybrid + multi-query + weighted RRF + cross-encoder + MMR + reorder), and generates grounded answers with Gemma 3 via Ollama.

**Assumes:** Notebook A (`04_propertylens_build_index.ipynb`) has already run successfully. This notebook never touches raw CSVs, never builds chunks, never upserts.

**Change from v4:** prediction tool removed. The `HybridClusterEnsemble` joblib load was the remaining memory hotspot after retrieval — dropping it removes ~1 GB of allocation pressure and 4 unnecessary Ollama calls per demo run. Retrieval + LLM answering still work exactly as before.

**Memory discipline:**
- Cross-encoder pinned to CPU (avoids MPS fighting with Gemma on Mac)
- No joblib model load
- Smoke test wrapped in try/except with RSS logged at each stage

![PropertyLens RAG inference pipeline](../../images/Screenshot%202026-04-17%20at%201.29.11%E2%80%AFPM.png)


### Install dependencies

In [6]:
# Inference-only deps. joblib removed since we no longer load the model bundle.
!pip install -q pinecone pinecone-text sentence-transformers transformers torch \
               ollama pandas numpy python-dotenv psutil

### Configuration

In [7]:
from __future__ import annotations
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# ── Pinecone (must match Notebook A) ─────────────────────────────────────────
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX   = "propertylens-rag"
NS_TRANSACTIONS  = "transactions"
NS_AMENITIES     = "amenities"
NS_XAI           = "xai"
NS_TRENDS        = "trends"

assert PINECONE_API_KEY, "Missing PINECONE_API_KEY in repo-root .env"

# ── Models ────────────────────────────────────────────────────────────────────
DENSE_MODEL_NAME = "BAAI/bge-m3"
RERANKER_MODEL   = "BAAI/bge-reranker-v2-m3"

# ── Retrieval knobs ───────────────────────────────────────────────────────────
TOP_K_RETRIEVAL = 50
TOP_K_RERANK    = 10
TOP_K_MMR       = 5
TOP_K_FINAL     = 5
MMR_LAMBDA      = 0.7
RRF_K           = 60
N_SUBQUERIES    = 3

# Source weights for weighted RRF — boosts amenity/xai chunks against the
# much-larger transactions pool.
SOURCE_WEIGHTS: dict[str, float] = {
    "transaction": 1.0,
    "amenity":     2.5,
    "trend":       1.0,
    "xai":         2.5,
}

# ── LLM ───────────────────────────────────────────────────────────────────────
OLLAMA_MODEL    = "gemma3"
OLLAMA_BASE_URL = "http://localhost:11434"

# ── Device pinning ────────────────────────────────────────────────────────────
# Pin cross-encoder to CPU. On Mac with Ollama running, MPS + Gemma + bi-encoder
# + cross-encoder compete for the same memory pool; CPU for CE is the single
# most effective stability fix.
CROSS_ENCODER_DEVICE = "cpu"

# ── Paths (must match Notebook A) ────────────────────────────────────────────
def find_repo_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()
    for p in [here, *here.parents]:
        if (p / "data").exists() and (p / "notebooks").exists():
            return p
    raise FileNotFoundError("Cannot find repo root.")

REPO_ROOT       = find_repo_root()
BM25_CACHE_PATH = REPO_ROOT / "notebooks" / "05_chatbot" / "bm25_encoder_v3.pkl"

print("Config loaded.")
print(f"  Pinecone index     : {PINECONE_INDEX}")
print(f"  BM25 cache path    : {BM25_CACHE_PATH}")
print(f"  CE device          : {CROSS_ENCODER_DEVICE}")
print(f"  Source weights     : {SOURCE_WEIGHTS}")


Config loaded.
  Pinecone index     : propertylens-rag
  BM25 cache path    : /Users/bhuvesh/Documents/PropertyLens/notebooks/05_chatbot/bm25_encoder_v3.pkl
  CE device          : cpu
  Source weights     : {'transaction': 1.0, 'amenity': 2.5, 'trend': 1.0, 'xai': 2.5}


### Memory helpers

`mem(label)` forces GC and prints RSS — watch it stage-by-stage.


In [8]:
from __future__ import annotations
import gc
import psutil

_PROC = psutil.Process(os.getpid())

def rss_mb() -> float:
    return _PROC.memory_info().rss / (1024 * 1024)

def mem(label: str) -> None:
    gc.collect()
    print(f"  [MEM] {label:<32s} RSS = {rss_mb():8.1f} MB")

mem("startup")


  [MEM] startup                          RSS =    997.5 MB


### Connect to Pinecone

No `create_index` — Notebook A should have populated it already. If the index is empty, retrieval will return zero chunks.


In [9]:
from __future__ import annotations
from pinecone import Pinecone

pc    = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX)

stats = index.describe_index_stats()
print(stats)
total = stats.get("total_vector_count") if isinstance(stats, dict) else getattr(stats, "total_vector_count", 0)
if not total:
    print("\n⚠ Pinecone index appears empty. Run Notebook A first.")


{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '279',
                                    'content-type': 'application/json',
                                    'date': 'Fri, 17 Apr 2026 07:30:51 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '40',
                                    'x-pinecone-request-latency-ms': '39',
                                    'x-pinecone-response-duration-ms': '42'}},
 'dimension': 1024,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'amenities': {'vector_count': 85},
                'transactions': {'vector_count': 1995},
                'trends': {'vector_count': 312},
                'xai': {'vector_count': 204}},
 'storageFullness': 0.0,
 'total_vector_count': 2596,
 'vector_type': 'dense'}


### Load encoders

Dense encoder (BGE-M3) on default device. BM25 loaded from cache — **fails loudly if the cache is missing** rather than silently refitting a different encoder than the one used at upsert.


In [10]:
from __future__ import annotations
import pickle
from sentence_transformers import SentenceTransformer
from pinecone_text.sparse import BM25Encoder


def load_bm25_from_cache(cache_path: Path) -> BM25Encoder:
    if not cache_path.exists():
        raise FileNotFoundError(
            f"BM25 cache not found: {cache_path}\n"
            f"Run Notebook A (04_propertylens_build_index.ipynb) first."
        )
    with cache_path.open("rb") as f:
        return pickle.load(f)


dense_encoder = SentenceTransformer(DENSE_MODEL_NAME)
bm25_encoder  = load_bm25_from_cache(BM25_CACHE_PATH)
print(f"Dense encoder : {DENSE_MODEL_NAME}")
print(f"BM25 encoder  : loaded from {BM25_CACHE_PATH}")
mem("after encoders loaded")


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 40041.33it/s]


Dense encoder : BAAI/bge-m3
BM25 encoder  : loaded from /Users/bhuvesh/Documents/PropertyLens/notebooks/05_chatbot/bm25_encoder_v3.pkl
  [MEM] after encoders loaded            RSS =   1289.2 MB


### Load cross-encoder (pinned to CPU)

Explicitly on CPU to sidestep MPS/CUDA contention with Ollama.


In [11]:
from __future__ import annotations
from typing import Any, Tuple
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer


def load_cross_encoder(model_name: str, device: str) -> Tuple[Any, Any]:
    """Load cross-encoder tokenizer and model, pinned to device."""
    tok   = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.to(device)
    model.eval()
    return tok, model


ce_tokenizer, ce_model = load_cross_encoder(RERANKER_MODEL, CROSS_ENCODER_DEVICE)
print(f"Cross-encoder loaded on device: {CROSS_ENCODER_DEVICE}")
mem("after cross-encoder loaded")


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 13588.23it/s]

Cross-encoder loaded on device: cpu
  [MEM] after cross-encoder loaded       RSS =   1469.7 MB


### NLP filter extraction + namespace routing

Lightweight pre-processing before retrieval:

- **Filter extraction:** Gemma extracts `town`, `flat_type`, `sale_year` from the query → Pinecone metadata filter on the transactions namespace only.
- **Namespace routing:** keyword-based selection of which namespaces to query. Always includes transactions.


In [12]:
from __future__ import annotations
import json, re
import ollama


def _set_ollama_host(base_url: str) -> None:
    os.environ["OLLAMA_HOST"] = base_url


def extract_filters_from_query(query: str, model: str = OLLAMA_MODEL) -> dict | None:
    """Use Gemma 3 to extract Pinecone metadata filters from a free-text query."""
    _set_ollama_host(OLLAMA_BASE_URL)
    prompt = f"""Extract structured fields from this Singapore HDB property query.
Return ONLY a valid JSON object with these optional keys:
  - "town": ALL CAPS HDB town e.g. "TAMPINES", "BEDOK", "SERANGOON"
  - "flat_type": one of "2 ROOM","3 ROOM","4 ROOM","5 ROOM","EXECUTIVE"
  - "sale_year": integer year if mentioned
Omit any field you are not sure about. Return {{}} if nothing is clear.
Return ONLY JSON, no explanation.

Query: {query}
"""
    try:
        response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
        raw      = (response.get("message") or {}).get("content", "{}")
        raw      = re.sub(r"```[\w]*", "", raw).strip()
        parsed   = json.loads(raw)
        return parsed if parsed else None
    except Exception as e:
        print(f"  Filter extraction failed: {e}")
        return None


def route_namespaces(query: str) -> list[str]:
    """Select Pinecone namespaces to query based on keywords."""
    q  = query.lower()
    ns = [NS_TRANSACTIONS]
    if any(kw in q for kw in ["mrt", "school", "mall", "hawker", "near", "amenity", "transport", "bus"]):
        ns.append(NS_AMENITIES)
    if any(kw in q for kw in ["trend", "rising", "falling", "increase", "decrease", "history",
                               "recent", "last year", "past", "over time", "appreciation"]):
        ns.append(NS_TRENDS)
    if any(kw in q for kw in ["explain", "shap", "feature", "why", "reason", "driver",
                               "factor", "importan", "predict", "model say"]):
        ns.append(NS_XAI)
    return ns


# Smoke test
test_q = "Is $580k fair for a 4-room in Tampines?"
print(f"Query      : {test_q}")
print(f"Filters    : {extract_filters_from_query(test_q)}")
print(f"Namespaces : {route_namespaces(test_q)}")


Query      : Is $580k fair for a 4-room in Tampines?
Filters    : {'town': 'TAMPINES', 'flat_type': '4 ROOM'}
Namespaces : ['transactions']


### Hybrid retrieval + source-weighted RRF


In [13]:
from __future__ import annotations
from typing import Any, Optional
import numpy as np


def _scale_sparse(sparse: dict, scale: float) -> dict:
    return {"indices": sparse["indices"], "values": [v * scale for v in sparse["values"]]}


def _hybrid_query(
    index,
    query: str,
    alpha: float,
    top_k: int,
    namespace: str,
    metadata_filter: Optional[dict] = None,
) -> list[dict[str, Any]]:
    """Single Pinecone hybrid query. alpha=1.0 → pure dense, 0.0 → pure sparse."""
    dense  = dense_encoder.encode(query, normalize_embeddings=True).tolist()
    dense  = (np.array(dense, dtype=np.float32) * float(alpha)).tolist()
    sparse = bm25_encoder.encode_queries([query])[0]
    sparse = _scale_sparse(sparse, 1.0 - float(alpha))
    res    = index.query(
        vector=dense, sparse_vector=sparse, top_k=int(top_k),
        namespace=namespace, include_metadata=True, filter=metadata_filter or None,
    )
    matches = res.get("matches") if isinstance(res, dict) else getattr(res, "matches", [])
    return [
        {"id": getattr(m, "id", m.get("id")),
         "score": getattr(m, "score", m.get("score")),
         "metadata": getattr(m, "metadata", m.get("metadata", {}))}
        for m in (matches or [])
    ]


def retrieve_from_namespace(
    index,
    query: str,
    namespace: str,
    top_k: int,
    metadata_filter: Optional[dict] = None,
) -> tuple[list[dict], list[dict]]:
    """Dense + sparse retrieval from one namespace."""
    dense  = _hybrid_query(index, query, alpha=1.0, top_k=top_k,
                           namespace=namespace, metadata_filter=metadata_filter)
    sparse = _hybrid_query(index, query, alpha=0.0, top_k=top_k,
                           namespace=namespace, metadata_filter=metadata_filter)
    return dense, sparse


def reciprocal_rank_fusion(
    ranked_lists: list[list[dict[str, Any]]],
    k: int = 60,
    source_weights: dict[str, float] | None = None,
) -> list[dict[str, Any]]:
    """
    Merge ranked lists using RRF with source-aware weights.

    score(d) = Σ  weight(source) × 1 / (k + rank_i(d))
    """
    weights = source_weights or SOURCE_WEIGHTS
    scores: dict[str, float] = {}
    best:   dict[str, dict]  = {}
    for lst in ranked_lists:
        for rank, r in enumerate(lst, start=1):
            rid = str(r.get("id", ""))
            if not rid:
                continue
            source = str((r.get("metadata") or {}).get("source", "transaction"))
            weight = weights.get(source, 1.0)
            scores[rid] = scores.get(rid, 0.0) + weight * (1.0 / (float(k) + float(rank)))
            if rid not in best:
                best[rid] = r
    fused = [{**best[rid], "rrf_score": sc} for rid, sc in scores.items()]
    fused.sort(key=lambda x: x.get("rrf_score", 0.0), reverse=True)
    return fused


print("Retrieval functions defined (source-weighted RRF active).")


Retrieval functions defined (source-weighted RRF active).


### Reranking funnel

Cross-encoder → MMR → lost-in-middle reorder. 50 → 10 → 5.


In [14]:
from __future__ import annotations


def _get_text(candidate: dict, field: str = "parent_text") -> str:
    md = candidate.get("metadata") or {}
    return str(md.get(field) or md.get("parent_text") or "")


def rerank_cross_encoder(
    query: str,
    candidates: list[dict[str, Any]],
    tokenizer: Any,
    model: Any,
    top_k: int,
    device: str = CROSS_ENCODER_DEVICE,
) -> list[dict[str, Any]]:
    """Score (query, passage) pairs with the cross-encoder; return top_k."""
    if not candidates:
        return []
    pairs  = [(query, _get_text(c)) for c in candidates]
    inputs = tokenizer(pairs, padding=True, truncation=True,
                       max_length=512, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits.squeeze(-1).tolist()
    if isinstance(logits, float):
        logits = [logits]
    scored = [{**c, "ce_score": float(s)} for c, s in zip(candidates, logits)]
    scored.sort(key=lambda x: x["ce_score"], reverse=True)
    return scored[:top_k]


def _cosine(u: np.ndarray, v: np.ndarray) -> float:
    denom = np.linalg.norm(u) * np.linalg.norm(v)
    return float(np.dot(u, v) / denom) if denom > 1e-12 else 0.0


def mmr_filter(
    candidates: list[dict[str, Any]],
    query: str,
    top_k: int,
    lambda_param: float = 0.7,
) -> list[dict[str, Any]]:
    """Select top_k diverse candidates via Maximal Marginal Relevance."""
    if not candidates:
        return []
    texts   = [_get_text(c) for c in candidates]
    doc_emb = np.asarray(
        dense_encoder.encode(texts, normalize_embeddings=True), dtype=np.float32
    )
    selected:  list[int] = []
    remaining: list[int] = list(range(len(candidates)))
    best0 = int(np.argmax([c.get("ce_score", -1e9) for c in candidates]))
    selected.append(best0)
    remaining.remove(best0)
    while remaining and len(selected) < int(top_k):
        best_idx, best_val = None, -1e18
        for i in remaining:
            rel     = float(candidates[i].get("ce_score", 0.0))
            max_sim = max(_cosine(doc_emb[i], doc_emb[j]) for j in selected)
            score   = lambda_param * rel - (1.0 - lambda_param) * max_sim
            if score > best_val:
                best_val, best_idx = score, i
        if best_idx is None:
            break
        selected.append(best_idx)
        remaining.remove(best_idx)
    return [candidates[i] for i in selected]


def reorder_for_context_window(
    candidates: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """Lost-in-the-middle mitigation: best first, second-best last."""
    if len(candidates) <= 2:
        return list(candidates)
    ordered = sorted(
        candidates,
        key=lambda x: x.get("ce_score", x.get("rrf_score", 0.0)),
        reverse=True,
    )
    return [ordered[0], *ordered[2:], ordered[1]]


print("Reranking funnel defined.")


Reranking funnel defined.


### Multi-query retrieval

Generate N sub-queries via Gemma, fan out across namespaces, fuse with weighted RRF.


In [15]:
from __future__ import annotations


def generate_subqueries(
    query: str,
    n: int = N_SUBQUERIES,
    model: str = OLLAMA_MODEL,
) -> list[str]:
    """Generate n reformulations of the query using Gemma 3."""
    _set_ollama_host(OLLAMA_BASE_URL)
    prompt = f"""You are a search query generator for Singapore HDB property data.
Generate {n} alternative search queries to help retrieve relevant data from a vector
database of HDB transactions, amenities, price trends, and SHAP features.
Return ONLY a numbered list. No explanations.

Original: {query}
"""
    try:
        response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
        raw      = (response.get("message") or {}).get("content", "")
        lines    = re.findall(r"^\s*\d+\.\s*(.+)$", raw, re.MULTILINE)
        return [l.strip().strip('"') for l in lines[:n]]
    except Exception as e:
        print(f"  Sub-query generation failed: {e}")
        return []


def multi_query_retrieve(
    query: str,
    index,
    namespaces: list[str],
    top_k: int,
    metadata_filter: dict | None = None,
) -> list[dict]:
    """Multi-query hybrid retrieval across all routed namespaces with weighted RRF."""
    all_queries = [query] + generate_subqueries(query)
    all_lists: list[list[dict]] = []
    for q in all_queries:
        for ns in namespaces:
            filt = metadata_filter if ns == NS_TRANSACTIONS else None
            dense, sparse = retrieve_from_namespace(index, q, ns, top_k, filt)
            all_lists.extend([dense, sparse])
    return reciprocal_rank_fusion(all_lists, k=RRF_K)


print("Multi-query retrieval defined.")


Multi-query retrieval defined.


### Full retrieval pipeline + defensive smoke test

Smoke test wrapped in try/except; `mem()` logged at each stage so any future crash tells you exactly which stage failed.


In [16]:
from __future__ import annotations


def retrieve_and_rerank(
    query: str,
    index,
    verbose: bool = False,
) -> list[dict]:
    """Full RAG retrieval pipeline for a free-text query."""
    if verbose: mem("  retr: start")
    metadata_filter = extract_filters_from_query(query)
    namespaces      = route_namespaces(query)
    if verbose: mem("  retr: after filter+route")

    fused = multi_query_retrieve(
        query=query, index=index, namespaces=namespaces,
        top_k=TOP_K_RETRIEVAL, metadata_filter=metadata_filter,
    )
    if verbose: mem(f"  retr: after fusion ({len(fused)} fused)")

    reranked = rerank_cross_encoder(
        query=query, candidates=fused[:TOP_K_RETRIEVAL],
        tokenizer=ce_tokenizer, model=ce_model, top_k=TOP_K_RERANK,
    )
    if verbose: mem(f"  retr: after CE rerank ({len(reranked)} ranked)")

    diverse = mmr_filter(
        candidates=reranked, query=query,
        top_k=TOP_K_MMR, lambda_param=MMR_LAMBDA,
    )
    if verbose: mem(f"  retr: after MMR ({len(diverse)} diverse)")

    final = reorder_for_context_window(diverse)[:TOP_K_FINAL]
    if verbose: mem("  retr: after reorder")
    return final


# ── Defensive smoke test ────────────────────────────────────────────────────
print("Smoke test (verbose mem tracking):")
try:
    smoke_ctx = retrieve_and_rerank(
        "Is $580k fair for a 4-room flat in Tampines?",
        index,
        verbose=True,
    )
    print(f"\n✓ Smoke test passed: {len(smoke_ctx)} chunks retrieved")
    for i, c in enumerate(smoke_ctx, 1):
        m = c.get("metadata") or {}
        print(f"  [{i}] {m.get('source')} | {m.get('town')} | "
              f"{m.get('flat_type','')} | {m.get('sale_year','')}")
except Exception as e:
    print(f"\n✗ Smoke test failed: {type(e).__name__}: {e}")
    import traceback; traceback.print_exc()

mem("after smoke test")


Smoke test (verbose mem tracking):
  [MEM]   retr: start                    RSS =   1174.0 MB
  [MEM]   retr: after filter+route       RSS =    942.3 MB
  [MEM]   retr: after fusion (59 fused)  RSS =   1043.7 MB
  [MEM]   retr: after CE rerank (10 ranked) RSS =   2827.8 MB
  [MEM]   retr: after MMR (5 diverse)    RSS =   2862.0 MB
  [MEM]   retr: after reorder            RSS =   2862.0 MB

✓ Smoke test passed: 5 chunks retrieved
  [1] transaction | TAMPINES | 4 ROOM | 2025
  [2] transaction | TAMPINES | 4 ROOM | 2022
  [3] transaction | TAMPINES | 4 ROOM | 2021
  [4] transaction | TAMPINES | 4 ROOM | 2023
  [5] transaction | TAMPINES | 4 ROOM | 2017
  [MEM] after smoke test                 RSS =   2862.0 MB


### Prompt builder + `generate_answer`

System prompt still mentions model predictions conditionally ("if given"), but the pipeline no longer computes one. The prompt builder handles `prediction_result=""` cleanly — no prediction section appears in the prompt when empty.


In [17]:
from __future__ import annotations


def build_system_prompt() -> str:
    """System prompt for Gemma 3."""
    return """You are a Singapore HDB property pricing assistant for PropertyLens.
Help buyers and sellers make informed decisions about HDB resale prices.

Rules:
1. Answer ONLY using the provided context. No outside knowledge.
2. Cite every specific claim with [Context N] labels.
3. If evidence is thin or contradictory, say so clearly.
4. Keep answers to 3-5 sentences unless detail is requested.
5. Give a Fair / Above market / Below market verdict ONLY for price fairness questions.
   For amenity, trend, or explanation questions, do NOT give a price verdict.
"""


def build_rag_prompt(
    query: str,
    context_chunks: list[dict[str, Any]],
    prediction_result: str = "",
) -> str:
    """Build the user-turn prompt with labelled context chunks."""
    parts = ["## Retrieved context"]
    for i, c in enumerate(context_chunks, 1):
        md_  = c.get("metadata") or {}
        txt  = str(md_.get("parent_text") or "").strip()
        hdr  = f"[Context {i}] source={md_.get('source')} town={md_.get('town')} year={md_.get('sale_year')}"
        parts.extend([hdr, txt, ""])
    if prediction_result:
        parts.extend(["## Model prediction", prediction_result, ""])
    parts.extend(["## Question", query])
    return "\n".join(parts).strip()


def generate_answer(
    query: str,
    context_chunks: list[dict[str, Any]],
    prediction_result: str = "",
    model: str = OLLAMA_MODEL,
) -> str:
    """Grounded RAG answer using Ollama + Gemma 3."""
    _set_ollama_host(OLLAMA_BASE_URL)
    try:
        response = ollama.chat(
            model=model,
            messages=[
                {"role": "system", "content": build_system_prompt()},
                {"role": "user",   "content": build_rag_prompt(query, context_chunks, prediction_result)},
            ],
        )
        return (response.get("message") or {}).get("content", "")
    except Exception as e:
        return f"[Ollama error] {type(e).__name__}: {e}"


print("Prompt builder and generate_answer defined.")


Prompt builder and generate_answer defined.


### End-to-end demo

Same 8 queries, each wrapped independently. No prediction tool call — retrieval + LLM answer only.


In [ ]:
from __future__ import annotations

import html

from IPython.display import Markdown, display

DEMO_QUERIES = [
    {"query": "Is $580k fair for a 4-room HDB in Tampines Street 81, Block 432?",
     "persona": "Buyer"},
    {"query": "What should I list my 5-room Bishan flat for given current market trends?",
     "persona": "Seller"},
    {"query": "Are HDB prices in Queenstown rising or falling over the last 3 years?",
     "persona": "Trends"},
    {"query": "What amenities are near Bedok North? Any MRT stations or schools?",
     "persona": "Amenities"},
    {"query": "Why did the model predict a high price for this Queenstown flat? What features drove it?",
     "persona": "XAI"},
    {"query": "Is $430,000 fair for a 3-room HDB at 1 Lorong Lew Lian Serangoon? "
              "64 sqm, 52 years lease remaining, lease started 1978, "
              "3 mins walk to Serangoon MRT.",
     "persona": "PropertyGuru listing"},
    {"query": "Should I buy a 4-room flat in Tampines or Bedok? "
              "Compare prices, trends, and nearby amenities.",
     "persona": "Cross-source comparison"},
    {"query": "The seller is asking $650k for a 5-room in Sengkang. "
              "What is a reasonable counter-offer based on recent sales?",
     "persona": "Negotiation"},
]

# Demo print colours (Markdown + HTML)
_COLOR_QUERY = "#c62828"
_COLOR_CTX = "#1565c0"
_COLOR_ANS = "#2e7d32"


def _md_section(title: str, body: str, color: str) -> None:
    """Render a titled block in Jupyter using Markdown (HTML), full-width coloured text."""
    safe_title = html.escape(title)
    safe_body = html.escape(body)
    display(
        Markdown(
            f"<div style=\"border-left:4px solid {color};margin:0.6em 0;padding:0.5em 0.75em;"
            f"background:#fafafa;color:{color}\">"
            f"<div style=\"font-weight:700;margin-bottom:0.4em\">{safe_title}</div>"
            f"<pre style=\"white-space:pre-wrap;font-family:inherit;font-size:0.95em;"
            f"margin:0;line-height:1.45;color:{color}\">{safe_body}</pre>"
            f"</div>"
        )
    )


def _format_chunk_block(i: int, c: dict) -> str:
    """Plain-text block for one retrieved chunk (full parent_text)."""
    m = c.get("metadata") or {}
    source = m.get("source", "")
    parts = [f"source={source}"]
    if m.get("town") is not None:
        parts.append(f"town={m.get('town')}")
    if m.get("flat_type") is not None:
        parts.append(f"flat_type={m.get('flat_type')}")
    if m.get("sale_year") is not None:
        parts.append(f"sale_year={m.get('sale_year')}")
    if m.get("amenity_type") is not None:
        parts.append(f"amenity_type={m.get('amenity_type')}")
    if m.get("xai_type") is not None:
        parts.append(f"xai_type={m.get('xai_type')}")
    rp = m.get("resale_price")
    if isinstance(rp, (int, float)):
        parts.append(f"resale_price=${int(rp):,}")
    if m.get("count") is not None:
        parts.append(f"count={m.get('count')}")

    header = f"----- Chunk {i} ({' | '.join(parts)}) -----"
    body = str(m.get("parent_text") or m.get("text") or "").strip()
    if body:
        return f"{header}\n(full text, {len(body)} chars)\n{body}"
    return f"{header}\n(no parent_text; raw metadata below)\n{m}"


def run_demo(demo: dict) -> None:
    """Run one demo query end-to-end. Isolated so failures don't cascade."""
    query = demo["query"]
    print(f"\n{'='*60}")
    try:
        metadata_filter = extract_filters_from_query(query)
        namespaces = route_namespaces(query)

        _md_section(
            f"Query — {demo['persona']}",
            query,
            _COLOR_QUERY,
        )
        print(f"  Filter     : {metadata_filter}")
        print(f"  Namespaces : {namespaces}")

        mem("before retrieve")
        ctx = retrieve_and_rerank(query, index)
        mem("after retrieve")

        ctx_body = "\n\n".join(_format_chunk_block(i, c) for i, c in enumerate(ctx, 1))
        _md_section(
            f"Context retrieved ({len(ctx)} chunks)",
            ctx_body if ctx_body else "(no chunks)",
            _COLOR_CTX,
        )

        mem("before generate_answer")
        answer = generate_answer(query, ctx)
        mem("after generate_answer")
        _md_section("Answer", answer, _COLOR_ANS)
    except Exception as e:
        print(f"\n  ✗ Demo failed: {type(e).__name__}: {e}")
        import traceback

        traceback.print_exc()
    print(f"{'='*60}")


for demo in DEMO_QUERIES:
    run_demo(demo)

mem("after demos")


<div style="border-left:4px solid #c62828;margin:0.6em 0;padding:0.5em 0.75em;background:#fafafa;color:#c62828"><div style="font-weight:700;margin-bottom:0.4em">Query — Buyer</div><pre style="white-space:pre-wrap;font-family:inherit;font-size:0.95em;margin:0;line-height:1.45;color:#c62828">Is $580k fair for a 4-room HDB in Tampines Street 81, Block 432?</pre></div>

  Filter     : {'town': 'TAMPINES', 'flat_type': '4 ROOM'}
  Namespaces : ['transactions']
  [MEM] before retrieve                  RSS =   2862.2 MB
  [MEM] after retrieve                   RSS =   3419.1 MB


<div style="border-left:4px solid #1565c0;margin:0.6em 0;padding:0.5em 0.75em;background:#fafafa;color:#1565c0"><div style="font-weight:700;margin-bottom:0.4em">Context retrieved (5 chunks)</div><pre style="white-space:pre-wrap;font-family:inherit;font-size:0.95em;margin:0;line-height:1.45;color:#1565c0">----- Chunk 1 (source=transaction | town=TAMPINES | flat_type=4 ROOM | sale_year=2025 | resale_price=$558,000) -----
(full text, 571 chars)
This is an HDB resale transaction in TAMPINES, year 2025.
Town: TAMPINES
Flat type: 4 ROOM
Storey band: unknown
Floor area (sqm): 84.0
Resale price (SGD): 558000
Approx PSF (SGD): 617.1

Nearby amenities (town-level):
Hawker Centres (3): HAWKER CENTRE (U/C), HAWKER CENTRE @ OUR TAMPINES HUB, TAMPINES ROUND MARKET AND FOOD CENTRE
Malls (4): TAMPINES MALL, TAMPINES 1, CENTURY SQUARE, ...
Mrt Stations (2): TAMPINES EAST MRT STATION (DT33), TAMPINES WEST MRT STATION (DT31)
Schools (12): GONGSHANG PRIMARY SCHOOL, SAINT HILDA&#x27;S PRIMARY SCHOOL, ANGSANA PRIMARY SCHOOL, ...

----- Chunk 2 (source=transaction | town=TAMPINES | flat_type=4 ROOM | sale_year=2017 | resale_price=$439,800) -----
(full text, 571 chars)
This is an HDB resale transaction in TAMPINES, year 2017.
Town: TAMPINES
Flat type: 4 ROOM
Storey band: unknown
Floor area (sqm): 89.0
Resale price (SGD): 439800
Approx PSF (SGD): 459.1

Nearby amenities (town-level):
Hawker Centres (3): HAWKER CENTRE (U/C), HAWKER CENTRE @ OUR TAMPINES HUB, TAMPINES ROUND MARKET AND FOOD CENTRE
Malls (4): TAMPINES MALL, TAMPINES 1, CENTURY SQUARE, ...
Mrt Stations (2): TAMPINES EAST MRT STATION (DT33), TAMPINES WEST MRT STATION (DT31)
Schools (12): GONGSHANG PRIMARY SCHOOL, SAINT HILDA&#x27;S PRIMARY SCHOOL, ANGSANA PRIMARY SCHOOL, ...

----- Chunk 3 (source=transaction | town=TAMPINES | flat_type=4 ROOM | sale_year=2017 | resale_price=$513,000) -----
(full text, 572 chars)
This is an HDB resale transaction in TAMPINES, year 2017.
Town: TAMPINES
Flat type: 4 ROOM
Storey band: unknown
Floor area (sqm): 104.0
Resale price (SGD): 513000
Approx PSF (SGD): 458.3

Nearby amenities (town-level):
Hawker Centres (3): HAWKER CENTRE (U/C), HAWKER CENTRE @ OUR TAMPINES HUB, TAMPINES ROUND MARKET AND FOOD CENTRE
Malls (4): TAMPINES MALL, TAMPINES 1, CENTURY SQUARE, ...
Mrt Stations (2): TAMPINES EAST MRT STATION (DT33), TAMPINES WEST MRT STATION (DT31)
Schools (12): GONGSHANG PRIMARY SCHOOL, SAINT HILDA&#x27;S PRIMARY SCHOOL, ANGSANA PRIMARY SCHOOL, ...

----- Chunk 4 (source=transaction | town=TAMPINES | flat_type=4 ROOM | sale_year=2017 | resale_price=$410,000) -----
(full text, 572 chars)
This is an HDB resale transaction in TAMPINES, year 2017.
Town: TAMPINES
Flat type: 4 ROOM
Storey band: unknown
Floor area (sqm): 104.0
Resale price (SGD): 410000
Approx PSF (SGD): 366.3

Nearby amenities (town-level):
Hawker Centres (3): HAWKER CENTRE (U/C), HAWKER CENTRE @ OUR TAMPINES HUB, TAMPINES ROUND MARKET AND FOOD CENTRE
Malls (4): TAMPINES MALL, TAMPINES 1, CENTURY SQUARE, ...
Mrt Stations (2): TAMPINES EAST MRT STATION (DT33), TAMPINES WEST MRT STATION (DT31)
Schools (12): GONGSHANG PRIMARY SCHOOL, SAINT HILDA&#x27;S PRIMARY SCHOOL, ANGSANA PRIMARY SCHOOL, ...

----- Chunk 5 (source=transaction | town=TAMPINES | flat_type=4 ROOM | sale_year=2017 | resale_price=$488,000) -----
(full text, 572 chars)
This is an HDB resale transaction in TAMPINES, year 2017.
Town: TAMPINES
Flat type: 4 ROOM
Storey band: unknown
Floor area (sqm): 104.0
Resale price (SGD): 488000
Approx PSF (SGD): 435.9

Nearby amenities (town-level):
Hawker Centres (3): HAWKER CENTRE (U/C), HAWKER CENTRE @ OUR TAMPINES HUB, TAMPINES ROUND MARKET AND FOOD CENTRE
Malls (4): TAMPINES MALL, TAMPINES 1, CENTURY SQUARE, ...
Mrt Stations (2): TAMPINES EAST MRT STATION (DT33), TAMPINES WEST MRT STATION (DT31)
Schools (12): GONGSHANG PRIMARY SCHOOL, SAINT HILDA&#x27;S PRIMARY SCHOOL, ANGSANA PRIMARY SCHOOL, ...</pre></div>

  [MEM] before generate_answer           RSS =   3419.1 MB
  [MEM] after generate_answer            RSS =   3419.1 MB


<div style="border-left:4px solid #2e7d32;margin:0.6em 0;padding:0.5em 0.75em;background:#fafafa;color:#2e7d32"><div style="font-weight:700;margin-bottom:0.4em">Answer</div><pre style="white-space:pre-wrap;font-family:inherit;font-size:0.95em;margin:0;line-height:1.45;color:#2e7d32">Based on the available data, there isn’t a transaction for Tampines Street 81, Block 432 [Context 1]. However, a similar 4-room HDB in Tampines with an area of 84.0 sqm sold for $558,000 in 2025 [Context 1]. The approximate PSF is $617.1. [Context 1]</pre></div>

...

<div style="border-left:4px solid #1565c0;margin:0.6em 0;padding:0.5em 0.75em;background:#fafafa;color:#1565c0"><div style="font-weight:700;margin-bottom:0.4em">Context retrieved (5 chunks)</div><pre style="white-space:pre-wrap;font-family:inherit;font-size:0.95em;margin:0;line-height:1.45;color:#1565c0">----- Chunk 1 (source=trend | town=QUEENSTOWN | sale_year=2020 | resale_price=$660,000) -----
(full text, 98 chars)
HDB resale trend for QUEENSTOWN, year 2020: median resale price SGD 660,000 across 3 transactions.

----- Chunk 2 (source=trend | town=QUEENSTOWN | sale_year=2017 | resale_price=$565,000) -----
(full text, 102 chars)
HDB resale trend for QUEENSTOWN, year 2017: median resale price SGD 565,000 across 3,144 transactions.

----- Chunk 3 (source=trend | town=QUEENSTOWN | sale_year=2018 | resale_price=$400,000) -----
(full text, 98 chars)
HDB resale trend for QUEENSTOWN, year 2018: median resale price SGD 400,000 across 3 transactions.

----- Chunk 4 (source=trend | town=QUEENSTOWN | sale_year=2024 | resale_price=$928,000) -----
(full text, 98 chars)
HDB resale trend for QUEENSTOWN, year 2024: median resale price SGD 928,000 across 3 transactions.

----- Chunk 5 (source=trend | town=QUEENSTOWN | sale_year=2015 | resale_price=$688,888) -----
(full text, 98 chars)
HDB resale trend for QUEENSTOWN, year 2015: median resale price SGD 688,888 across 3 transactions.</pre></div>

  [MEM] before generate_answer           RSS =   3513.4 MB
  [MEM] after generate_answer            RSS =   3512.5 MB


<div style="border-left:4px solid #2e7d32;margin:0.6em 0;padding:0.5em 0.75em;background:#fafafa;color:#2e7d32"><div style="font-weight:700;margin-bottom:0.4em">Answer</div><pre style="white-space:pre-wrap;font-family:inherit;font-size:0.95em;margin:0;line-height:1.45;color:#2e7d32">Based on the provided data, HDB prices in Queenstown have been rising over the last three years [Context 1, Context 4, Context 5]. In 2020, the median resale price was SGD 660,000 [Context 1], while in 2024 it reached SGD 928,000 [Context 4]. The trend from 2017 to 2018 also shows an increase [Context 2, Context 3].</pre></div>

<div style="border-left:4px solid #c62828;margin:0.6em 0;padding:0.5em 0.75em;background:#fafafa;color:#c62828"><div style="font-weight:700;margin-bottom:0.4em">Query — Amenities</div><pre style="white-space:pre-wrap;font-family:inherit;font-size:0.95em;margin:0;line-height:1.45;color:#c62828">What amenities are near Bedok North? Any MRT stations or schools?</pre></div>

  Filter     : {'town': 'BEDOK'}
  Namespaces : ['transactions', 'amenities']
  [MEM] before retrieve                  RSS =   3512.5 MB


### Notes

- **Prediction tool removed** in v4.1. Retrieval + LLM answering unchanged. If you need price prediction, call the `HybridClusterEnsemble` from a separate script or a smaller notebook where it's the only thing loaded.
- **Restarting the kernel is cheap:** no CSVs, no chunk building. Re-running takes ~30 seconds.
- **`mem()` checkpoints around each demo sub-step** let you see memory behaviour per query. If the kernel still dies, the last printed `[MEM]` line tells you exactly which stage killed it.
- **If RSS climbs past ~8 GB on a 16 GB Mac during demos:** try `ollama stop gemma3` between sessions, or reduce `TOP_K_RERANK` to 6.
- **BM25 cache is required** — if `bm25_encoder_v3.pkl` is missing, this notebook fails at the "load encoders" cell. That's deliberate.
